<a href="https://www.kaggle.com/code/srikanthmachiraju/raft-finetuning-slm-inference?scriptVersionId=317128896" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Evaluate the Fine-Tuned RAFT SLM (Kaggle)

This is an inference notebook for RAFT finetuned SLM, for training notebook please visit: https://www.kaggle.com/code/srikanthmachiraju/raft-finetuning-slm/

This notebook runs on a **Kaggle GPU** instance (T4) and:

1. Loads the fine-tuned `llama-3.2-1B-Instruct` adapter from the Hugging Face Hub via Unsloth (4-bit).
2. Loads the held-out RAFT test split (`test.jsonl`).
3. Generates an answer for each record using the **same** chat template / system prompt used at training time.
4. Saves predictions to `slm_predictions.csv`.

## 0. Install dependencies

Mirrors the install block in `raft-finetuning-slm.ipynb` so the same Unsloth / Transformers versions are loaded on the Kaggle GPU. LlamaIndex Azure-OpenAI extras are added for the optional evaluation step.

In [1]:
%%capture
import os, subprocess
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
!pip install --upgrade -qqq uv
try:
    import numpy, PIL
    _numpy = f"numpy=={numpy.__version__}"
    _pil   = f"pillow=={PIL.__version__}"
except Exception:
    _numpy, _pil = "numpy", "pillow"
try:
    is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except Exception:
    is_t4 = False
_vllm, _triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.15.1", "triton")
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
!uv pip install -qqq transformers==4.56.2
!uv pip install -qqq --no-deps trl==0.22.2
!uv pip install -qqq llama-index llama-index-llms-azure-openai pandas tqdm

## 1. Configuration

Update these for your run. Defaults match the model id pushed by the fine-tuning notebook.

In [2]:
# --- Model & data ---
HF_MODEL_ID    = "sriksmachi/llama32_1bn_instruct_raft"  # fine-tuned model on HF Hub
MAX_SEQ_LEN    = 2048
MAX_NEW_TOKENS = 512
TEMPERATURE    = 0.0

# Path to the held-out test split. On Kaggle, attach this repo as a dataset and adjust the path.
# Two common options:
#   1) Add the repo as a Kaggle dataset:  /kaggle/input/<dataset-slug>/data/training_data_raft/test.jsonl
#   2) Upload test.jsonl directly:         /kaggle/input/raft-test/test.jsonl
TEST_JSONL = "/kaggle/input/datasets/srikanthmachiraju/raft-finetuning-dataset/test.jsonl"

# --- Output paths (Kaggle working dir is persisted as session output) ---
PRED_CSV = "/kaggle/working/slm_predictions.csv"
EVAL_CSV = "/kaggle/working/slm_eval_results.csv"
BASELINE_CSV = "/kaggle/working/slm_baseline_predictions.csv"

# --- Optional: cap sample count for a quick smoke run ---
LIMIT = None  # e.g., 20 to test the full pipeline quickly

# --- Run the LlamaIndex evaluation step? ---
RUN_EVAL = True

## 2. Load the fine-tuned model

Unsloth merges the LoRA adapter back into the 4-bit base for fast inference. The same `llama-3.1` chat template used during fine-tuning is applied here.

In [3]:
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = HF_MODEL_ID,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
    dtype          = None,
)

baseline_model, baseline_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
    dtype          = None,
)


tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

# Switch to inference mode (Unsloth's 2x faster path)
FastLanguageModel.for_inference(model)
print("Model loaded:", HF_MODEL_ID)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-06 17:56:50.498487: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778090210.949937      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778090211.064163      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778090212.066559      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778090212.066617      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778090212.066620      23 computation_placer.cc:177] computation placer alr

INFO 05-06 17:57:44 [__init__.py:244] Automatically detected platform cuda.
ERROR 05-06 17:57:49 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/22.6M [00:00<?, ?B/s]

Unsloth 2026.5.2 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded: sriksmachi/llama32_1bn_instruct_raft


## 3. Load the test split

In [4]:
import json
from pathlib import Path

def load_jsonl(path, limit=None):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    if limit is not None:
        rows = rows[:limit]
    return rows

records = load_jsonl(TEST_JSONL, LIMIT)
print(f"Loaded {len(records)} test record(s) from {TEST_JSONL}")
print("First record keys:", list(records[0].keys()))

Loaded 64 test record(s) from /kaggle/input/datasets/srikanthmachiraju/raft-finetuning-dataset/test.jsonl
First record keys: ['id', 'type', 'question', 'context', 'oracle_context', 'cot_answer', 'instruction']


## 4. Generate answers with the fine-tuned SLM

We mirror the **exact** chat-style prompt used during fine-tuning (`raft-finetuning-slm.ipynb`):

- `system`  → the fine-tuning system prompt
- `user`    → `"<Retrieved Documents>: \n{instruction}"` (the `instruction` field already contains the documents + the question)
- `assistant` → generated by the model (CoT + `<ANSWER>:`)

In [5]:
from tqdm.auto import tqdm

_SYSTEM_PROMPT = "You are a helpful assistant that answers questions using the provided context ONLY. \
Given the question, context and answer above, provide a logical reasoning for that answer. Please use the format of: \
### Step-by-step reasoning: \
##begin_quote## [Relevant text 1] ##end_quote## \
##begin_quote## [Relevant text 2] ##end_quote## \
<ANSWER>Answer here...</ANSWER>" 

@torch.inference_mode()
def generate_answer(instruction: str) -> str:
    messages = [
        {"role": "system", "content": _SYSTEM_PROMPT},
        {"role": "user",   "content": f"<Retrieved Documents>: \n{instruction}"},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize             = True,
        add_generation_prompt= True,
        return_tensors       = "pt",
    ).to(model.device)
    attention_mask = torch.ones_like(input_ids, device=model.device)

    out = model.generate(
        input_ids       = input_ids,
        attention_mask  = attention_mask,
        max_new_tokens  = MAX_NEW_TOKENS,
        do_sample       = TEMPERATURE > 0,
        temperature     = TEMPERATURE if TEMPERATURE > 0 else 1.0,
        use_cache       = True,
        pad_token_id    = tokenizer.eos_token_id,
    )
    # Slice off the prompt tokens, decode the new tokens only.
    new_tokens = out[0, input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

import pandas as pd

predictions = []
for r in tqdm(records, desc="SLM inference", unit="sample"):
    instr = r.get("instruction") or ""
    if not instr:
        continue
    try:
        ans = generate_answer(instr)
    except Exception as e:
        print("generation failed:", e)
        ans = ""
    predictions.append({
        "model_name": HF_MODEL_ID,
        "id":                r.get("id"),
        "type": r.get("type"),
        "question":          r.get("question", ""),
        "context":       instr,
        "model_answer":      ans,
        "reference_answer":  r.get("cot_answer", "")
    })

pred_df = pd.DataFrame(predictions)
Path(PRED_CSV).parent.mkdir(parents=True, exist_ok=True)
pred_df.to_csv(PRED_CSV, index=False)
print(f"Saved {len(pred_df)} predictions to {PRED_CSV}")
pred_df[["question", "model_answer"]].head(3)

SLM inference:   0%|          | 0/64 [00:00<?, ?sample/s]

Saved 64 predictions to /kaggle/working/slm_predictions.csv


,question,model_answer
0,Where can I access the Privacy/GDPR section in...,### Step-by-Step Reasoning:\n\n1. **Understand...
1,How can someone learn to access the CORE Compl...,### Step-by-Step Reasoning:\n\n1. **Understand...
2,What features does the AI Agent Studio offer f...,### Step-by-Step Reasoning:\n\n1. **Understand...


In [6]:
from tqdm.auto import tqdm

@torch.inference_mode()
def generate_answer(instruction: str) -> str:
    messages = [
        {"role": "system", "content": _SYSTEM_PROMPT},
        {"role": "user",   "content": f"<Retrieved Documents>: \n{instruction}"},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize             = True,
        add_generation_prompt= True,
        return_tensors       = "pt",
    ).to(baseline_model.device)
    attention_mask = torch.ones_like(input_ids, device=baseline_model.device)

    out = baseline_model.generate(
        input_ids       = input_ids,
        attention_mask  = attention_mask,
        max_new_tokens  = MAX_NEW_TOKENS,
        do_sample       = TEMPERATURE > 0,
        temperature     = TEMPERATURE if TEMPERATURE > 0 else 1.0,
        use_cache       = True,
        pad_token_id    = tokenizer.eos_token_id,
    )
    # Slice off the prompt tokens, decode the new tokens only.
    new_tokens = out[0, input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

import pandas as pd

baseline_predictions = []
for r in tqdm(records, desc="SLM inference", unit="sample"):
    instr = r.get("instruction") or ""
    if not instr:
        continue
    try:
        ans = generate_answer(instr)
    except Exception as e:
        print("generation failed:", e)
        ans = ""
    baseline_predictions.append({
        "model_name": "unsloth/Llama-3.2-1B-Instruct",
        "id":                r.get("id"),
        "type": r.get("type"),
        "question":          r.get("question", ""),
        "context":       instr,
        "model_answer":      ans,
        "reference_answer":  r.get("cot_answer", "")
    })

baseline_pred_df = pd.DataFrame(baseline_predictions)
Path(BASELINE_CSV).parent.mkdir(parents=True, exist_ok=True)
baseline_pred_df.to_csv(BASELINE_CSV, index=False)
print(f"Saved {len(baseline_pred_df)} predictions to {BASELINE_CSV}")
pred_df[["question", "model_answer"]].head(3)

SLM inference:   0%|          | 0/64 [00:00<?, ?sample/s]

Saved 64 predictions to /kaggle/working/slm_baseline_predictions.csv


,question,model_answer
0,Where can I access the Privacy/GDPR section in...,### Step-by-Step Reasoning:\n\n1. **Understand...
1,How can someone learn to access the CORE Compl...,### Step-by-Step Reasoning:\n\n1. **Understand...
2,What features does the AI Agent Studio offer f...,### Step-by-Step Reasoning:\n\n1. **Understand...


## 6. Done

Outputs in `/kaggle/working/`:

- `slm_predictions.csv` — per-sample model answers. Download and save locally for evaluation